# 05 — Robustness Checks (Appendix)

Three robustness appendices, each controlled by a toggle in `src/config.py`:

| Toggle | Content |
|---|---|
| `RUN_APPENDIX_FMB` | Fama-MacBeth with Newey-West SEs |
| `RUN_APPENDIX_ALT_WINDOWS` | Alternative event windows `[-2,+2]` and `[-1,+5]` |
| `RUN_APPENDIX_IMPKURT` | Implied kurtosis as tail-risk proxy (KR) |

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.config import (
    DATA_PATH, OUT_DIR, PRE_DAY, POST_DAY, MAIN_CLUSTER_VAR,
    FMB_DATE_COL, MIN_CS_OBS, NW_LAGS, ALT_WINDOWS,
    BASE_CONTROLS, RICH_OPTION_CONTROLS, SHOCK_PROXY,
    RUN_APPENDIX_FMB, RUN_APPENDIX_ALT_WINDOWS, RUN_APPENDIX_IMPKURT,
)
from src.data_utils import load_panel, build_event_level_dataset, safe_log
from src.stats import run_clustered_pooled_ols, run_fama_macbeth, available_controls

TABLES = OUT_DIR / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)

df    = load_panel(DATA_PATH)
event, _ = build_event_level_dataset(df, pre_day=PRE_DAY, post_day=POST_DAY)

base  = available_controls(event, BASE_CONTROLS)
rich  = available_controls(event, RICH_OPTION_CONTROLS)
shock = available_controls(event, SHOCK_PROXY)
print('Event obs:', len(event))

Event obs: 11468


## A — Fama-MacBeth

In [2]:
if RUN_APPENDIX_FMB:
    fmb_specs = {
        'H1_FMB_main':  ('SR',              ['PreSkew',    'LN_VIX_pre'] + base + rich),
        'H2_FMB_main':  ('UR',              ['LN_VIX_pre', 'PreSkew', 'PreIV'] + [c for c in base if c != 'PreIV'] + rich),
        'H2_FMB_norm':  ('UR_norm_by_PreIV',['LN_VIX_pre', 'PreSkew', 'PreIV'] + [c for c in base if c != 'PreIV'] + rich),
        'H2_FMB_IVpost':('IV_post',         ['LN_VIX_pre', 'PreSkew', 'PreIV'] + [c for c in base if c != 'PreIV'] + rich),
    }

    fmb_summaries = []
    for spec_name, (depvar, regressors) in fmb_specs.items():
        regs = list(dict.fromkeys([r for r in regressors if r in event.columns]))
        summary, coef_ts = run_fama_macbeth(
            event, depvar=depvar, regressors=regs,
            date_col=FMB_DATE_COL, min_obs=MIN_CS_OBS,
            lags=NW_LAGS, spec_name=spec_name,
        )
        fmb_summaries.append(summary)
        coef_ts.to_csv(TABLES / f'fmb_ts_{spec_name}.csv', index=False)

    fmb_all = pd.concat(fmb_summaries, ignore_index=True)
    fmb_all.to_csv(TABLES / 'fama_macbeth_robustness.csv', index=False)
    print('Saved: fama_macbeth_robustness.csv')
    display(fmb_all)
else:
    print('Skipped (RUN_APPENDIX_FMB=False)')

Saved: fama_macbeth_robustness.csv


,spec,depvar,term,coef_mean,t_stat_NW,T,avg_cs_N,avg_cs_adj_R2
0,H1_FMB_main,SR,const,-0.004618,-0.088848,98,111.714286,0.321983
1,H1_FMB_main,SR,PreSkew,0.652414,23.132264,98,111.714286,0.321983
2,H1_FMB_main,SR,LN_VIX_pre,-0.035288,-2.149282,98,111.714286,0.321983
3,H1_FMB_main,SR,PreIV,0.041865,8.930182,98,111.714286,0.321983
4,H1_FMB_main,SR,LN_MRKCAP_pre,-0.005952,-3.982924,98,111.714286,0.321983
5,H1_FMB_main,SR,ret_pre,-0.018081,-0.375451,98,111.714286,0.321983
6,H1_FMB_main,SR,LN_VOL_pre,0.007947,4.318408,98,111.714286,0.321983
7,H1_FMB_main,SR,LN_PRC_pre,0.005013,2.367866,98,111.714286,0.321983
8,H1_FMB_main,SR,Dispersion_pre,-0.012071,-0.497414,98,111.714286,0.321983
9,H1_FMB_main,SR,LN_PC_OI_pre,0.000647,1.624044,98,111.714286,0.321983


## B — Alternative Event Windows

In [3]:
if RUN_APPENDIX_ALT_WINDOWS:
    alt_rows, alt_meta = [], []

    for pre_d, post_d in ALT_WINDOWS:
        alt_event, _ = build_event_level_dataset(df, pre_day=pre_d, post_day=post_d)
        ab = available_controls(alt_event, BASE_CONTROLS)
        ar = available_controls(alt_event, RICH_OPTION_CONTROLS)
        ash = available_controls(alt_event, SHOCK_PROXY)

        alt_specs = {
            f'H1_{pre_d}_{post_d}': ('SR', ['PreSkew', 'LN_VIX_pre'] + ab + ar),
            f'H2_{pre_d}_{post_d}': ('UR', ['LN_VIX_pre', 'PreSkew', 'PreIV'] + [c for c in ab if c != 'PreIV'] + ar + ash),
        }
        for spec_name, (depvar, regressors) in alt_specs.items():
            regs = list(dict.fromkeys([r for r in regressors if r in alt_event.columns]))
            coef, meta, _ = run_clustered_pooled_ols(
                alt_event, depvar, regs, cluster_var=MAIN_CLUSTER_VAR, spec_name=spec_name
            )
            coef['window'] = f'[{pre_d},{post_d}]'
            meta['window'] = f'[{pre_d},{post_d}]'
            alt_rows.append(coef)
            alt_meta.append(meta)

    pd.concat(alt_rows, ignore_index=True).to_csv(TABLES / 'appendix_alt_window_results.csv', index=False)
    pd.concat(alt_meta, ignore_index=True).to_csv(TABLES / 'appendix_alt_window_meta.csv',    index=False)
    print('Saved: appendix_alt_window_results.csv, appendix_alt_window_meta.csv')
else:
    print('Skipped (RUN_APPENDIX_ALT_WINDOWS=False)')

Saved: appendix_alt_window_results.csv, appendix_alt_window_meta.csv


## C — Implied Kurtosis as Tail-Risk Proxy

In [4]:
if RUN_APPENDIX_IMPKURT:
    pre  = df.loc[df['rel_day'] == -1].copy()
    post = df.loc[df['rel_day'] ==  1].copy()

    pre_keep = [
        'event_id', 'secid', 'IMPKURT', 'LN_VIX', 'LN_IMPVOL', 'LN_MRKCAP',
        'ret', 'vol', 'prc', 'saleq', 'atq', 'SIC2', 'QUARTER_FE',
        'Dispersion', 'LN_PC_OI', 'LN_PC_VLM', 'LN_TOTALVAR', 'LN_EXPTIME',
    ]
    pre  = pre[[c for c in pre_keep if c in pre.columns]]
    post = post[[c for c in ['event_id', 'IMPKURT', 'ret'] if c in post.columns]]

    kurt_event = pre.merge(post, on='event_id', suffixes=('_pre', '_post'))
    kurt_event['KR']           = -(kurt_event['IMPKURT_post'] - kurt_event['IMPKURT_pre'])
    kurt_event['PreKurt']      = kurt_event['IMPKURT_pre']
    kurt_event['PreIV']        = kurt_event['LN_IMPVOL']
    kurt_event['LN_VIX_pre']   = kurt_event['LN_VIX']
    kurt_event['LN_MRKCAP_pre']= kurt_event['LN_MRKCAP']
    kurt_event['ret_pre']      = kurt_event['ret_pre']
    kurt_event['Abs_Ret']      = kurt_event['ret_post'].abs()
    kurt_event['LN_VOL_pre']   = safe_log(kurt_event['vol'],       lower=1)
    kurt_event['LN_PRC_pre']   = safe_log(kurt_event['prc'].abs(), lower=0.01)
    kurt_event['event_quarter']= kurt_event['QUARTER_FE'].astype(str)
    kurt_event['SIC2_pre']     = kurt_event['SIC2']

    for src_col, dst_col in [
        ('Dispersion',   'Dispersion_pre'),
        ('LN_PC_OI',     'LN_PC_OI_pre'),
        ('LN_PC_VLM',    'LN_PC_VLM_pre'),
        ('LN_TOTALVAR',  'LN_TOTALVAR_pre'),
        ('LN_EXPTIME',   'LN_EXPTIME_pre'),
    ]:
        if src_col in kurt_event.columns:
            kurt_event[dst_col] = kurt_event[src_col]

    kurt_regs = [
        'PreKurt', 'LN_VIX_pre', 'PreIV', 'LN_MRKCAP_pre',
        'ret_pre', 'LN_VOL_pre', 'LN_PRC_pre', 'Abs_Ret',
    ] + [c for c in [
        'Dispersion_pre', 'LN_PC_OI_pre', 'LN_PC_VLM_pre',
        'LN_TOTALVAR_pre', 'LN_EXPTIME_pre',
    ] if c in kurt_event.columns]

    coef, meta, _ = run_clustered_pooled_ols(
        kurt_event, 'KR', kurt_regs, cluster_var='secid', spec_name='H1_alt_tailrisk'
    )
    coef.to_csv(TABLES / 'h1_impkurt_robustness.csv', index=False)
    meta.to_csv(TABLES / 'h1_impkurt_meta.csv',       index=False)
    print('Saved: h1_impkurt_robustness.csv, h1_impkurt_meta.csv')
    display(coef)
else:
    print('Skipped (RUN_APPENDIX_IMPKURT=False)')

Saved: h1_impkurt_robustness.csv, h1_impkurt_meta.csv


,spec,depvar,term,coef,std_err,t_stat,p_value
0,H1_alt_tailrisk,KR,PreKurt,0.882001,0.013639,64.665273,0.000000
1,H1_alt_tailrisk,KR,LN_VIX_pre,-0.002655,0.002847,-0.932533,0.351061
2,H1_alt_tailrisk,KR,PreIV,-0.004208,0.002602,-1.617077,0.105862
3,H1_alt_tailrisk,KR,LN_MRKCAP_pre,0.003069,0.000932,3.292013,0.000995
4,H1_alt_tailrisk,KR,ret_pre,0.005411,0.017800,0.303973,0.761149
5,H1_alt_tailrisk,KR,LN_VOL_pre,-0.001460,0.000974,-1.498370,0.134037
6,H1_alt_tailrisk,KR,LN_PRC_pre,-0.000189,0.001031,-0.183580,0.854343
7,H1_alt_tailrisk,KR,Abs_Ret,-0.029635,0.023852,-1.242422,0.214081
8,H1_alt_tailrisk,KR,Dispersion_pre,0.001602,0.001611,0.994179,0.320136
9,H1_alt_tailrisk,KR,LN_PC_OI_pre,-0.000024,0.000169,-0.142319,0.886828
